<a href="https://colab.research.google.com/github/SergiSama/UIC-CRB1-2026-2027/blob/main/m4_segmentation_counting_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4 · Notebook 2 — Segmentation & Counting Cells (OpenCV)
### Computing, Robotics & Bionics · companion to A. Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*

The capstone of the refresher series: a complete classical pipeline that **segments and counts cells**,
scored against ground truth. We build it on a synthetic microscopy image whose true count we *know*
(offline, always runs), then point it at real **Broad Institute BBBC** images, whose cell count is encoded
in the filename. Where classical vision runs out, the next step is a convolutional network (Géron Ch. 12).

Run in **Google Colab** (*Runtime → Run all*). OpenCV is pre-installed; the synthetic path is fully
offline, and the BBBC cell is guarded.

**Contents**
1. A synthetic microscopy image (known count)
2. Thresholding and Otsu
3. Morphology: cleaning the mask
4. Contours, connected components and moments
5. The full counting pipeline (scored vs ground truth)
6. Optional: real Broad BBBC images

Most sections end with an **Exercise**; run the **Solution** cell to check.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
rng = np.random.default_rng(7)
print("OpenCV", cv2.__version__)

---
## 1 · A synthetic microscopy image (known count)

We place a known number of bright "cells" on a dark background, with soft edges and noise — a stand-in for
a fluorescence image. Because *we* set the count, we can score the pipeline exactly (and it runs with no
download).

In [ ]:
def make_cells(n_rows=5, n_cols=5, size=512, r_mean=16, jitter=22, seed=7):
    rng = np.random.default_rng(seed)
    img = np.zeros((size, size), np.uint8)
    centers = []
    step = size // (n_rows + 1)
    for i in range(1, n_rows+1):
        for j in range(1, n_cols+1):
            cy = i*step + int(rng.integers(-jitter, jitter))
            cx = j*step + int(rng.integers(-jitter, jitter))
            r = int(r_mean + rng.integers(-4, 5))
            cv2.circle(img, (cx, cy), r, int(rng.integers(170, 255)), -1)
            centers.append((cx, cy))
    img = cv2.GaussianBlur(img, (7, 7), 3)                    # soft edges
    img = np.clip(img + rng.normal(0, 8, img.shape), 0, 255).astype(np.uint8)  # noise
    return img, len(centers)

image, true_count = make_cells()
print("true cell count:", true_count)
plt.figure(figsize=(5,5)); plt.imshow(image, cmap="gray")
plt.title(f"synthetic microscopy image ({true_count} cells)"); plt.axis("off"); plt.show()

---
## 2 · Thresholding and Otsu

To separate cells from background we threshold. A fixed cutoff is brittle; **Otsu's method** picks the
threshold automatically as the histogram valley between the two populations.

In [ ]:
otsu_t, mask = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print("Otsu threshold chosen:", otsu_t)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].imshow(image, cmap="gray"); ax[0].set_title("image"); ax[0].axis("off")
ax[1].hist(image.ravel(), bins=64, color="steelblue"); ax[1].axvline(otsu_t, color="crimson", ls="--")
ax[1].set_title(f"histogram (Otsu={otsu_t:.0f})"); ax[1].set_xlim(0,255)
ax[2].imshow(mask, cmap="gray"); ax[2].set_title("binary mask"); ax[2].axis("off")
plt.show()

---
## 3 · Morphology: cleaning the mask

Morphological **opening** (erode then dilate) removes small specks; **closing** (dilate then erode) seals
holes inside objects. We add some debris first to show the cleanup.

In [ ]:
# sprinkle small noise specks into the mask to simulate debris
noisy_mask = mask.copy()
for _ in range(40):
    y, x = rng.integers(0, mask.shape[0]), rng.integers(0, mask.shape[1])
    cv2.circle(noisy_mask, (int(x), int(y)), 2, 255, -1)

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
opened = cv2.morphologyEx(noisy_mask, cv2.MORPH_OPEN, kernel)     # removes specks
clean  = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)        # seals holes

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, img, t in zip(ax, [noisy_mask, opened, clean], ["mask + debris", "after opening", "after closing"]):
    a.imshow(img, cmap="gray"); a.set_title(t); a.axis("off")
plt.show()

---
## 4 · Contours, connected components and moments

A clean mask is segmented into objects. **Connected-component** labelling counts blobs; **contours** trace
outlines; **moments** give each object's area and centroid.

In [ ]:
# connected components with stats (area lets us reject debris)
n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(clean)
areas = stats[1:, cv2.CC_STAT_AREA]            # skip label 0 (background)
print("raw components (excl. background):", n_labels - 1)
print("areas:", np.sort(areas)[:8], "...")

# contours + moments for the centroid of the first real cell
contours, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
big = [c for c in contours if cv2.contourArea(c) > 80]
M = cv2.moments(big[0])
cx, cy = M["m10"]/M["m00"], M["m01"]/M["m00"]
print(f"first cell: area={cv2.contourArea(big[0]):.0f}, centroid=({cx:.0f},{cy:.0f})")

---
## 5 · The full counting pipeline (scored vs ground truth)

We assemble the steps into one function — **blur → Otsu → morphology → components → area filter → count** —
and score it against the known count.

In [ ]:
def count_cells(img, min_area=80):
    blur = cv2.GaussianBlur(img, (5, 5), 1)
    _, m = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)
    n, lab, stats, cent = cv2.connectedComponentsWithStats(m)
    keep = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= min_area]  # reject debris
    return len(keep), cent[keep], m

count, centers, final_mask = count_cells(image)
print(f"counted {count} cells | ground truth {true_count} | "
      f"{'CORRECT' if count == true_count else 'off by ' + str(count - true_count)}")

# overlay detected centroids
overlay = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
for (x, y) in centers:
    cv2.circle(overlay, (int(x), int(y)), 18, (255, 0, 0), 2)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(final_mask, cmap="gray"); ax[0].set_title("final mask"); ax[0].axis("off")
ax[1].imshow(overlay); ax[1].set_title(f"detected {count} cells"); ax[1].axis("off")
plt.show()

**Exercise 5.** Generate a *denser* image with `make_cells(n_rows=7, n_cols=7)` and run `count_cells`
on it. Does the count still match? If cells touch, what step of the pipeline fails, and what would fix it
(hint: watershed, or a learned segmenter)?

> 🤖 *Gemini tip:* "Explain why connected-component counting undercounts when objects touch, and how a distance transform followed by watershed separates them."

In [ ]:
# Your code here

---
## 6 · Optional: real Broad BBBC images

The **Broad Bioimage Benchmark Collection** provides microscopy images with ground truth. In **BBBC005**
the true cell count is encoded in each filename (the number after `C`, e.g. `..._C18_...` = 18 cells), so
the same `count_cells` pipeline can be scored on real data.

**Where to get it (in Colab):**
- Full set (~1.8 GB zip): `https://data.broadinstitute.org/bbbc/BBBC005/BBBC005_v1_images.zip`
- Lighter subset (recommended for class): the Kaggle dataset *"Synthetic Cell Images and Masks (BBBC005 v1)"*.

The cell below first demonstrates the filename → count convention offline, then shows the exact loop to run
once you have images locally (guarded, so it never breaks).

In [ ]:
import re
def count_from_name(fname):
    m = re.search(r'_C(\d+)_', fname)            # BBBC005 encodes the count as _C<n>_
    return int(m.group(1)) if m else None

# Offline demo of the convention on a real BBBC005 filename:
example = "SIMCEPImages_A05_C18_F1_s05_w1.TIF"
print(f"{example}  ->  ground-truth count = {count_from_name(example)}")

In [ ]:
# Run on real BBBC005 images once you have downloaded/unzipped them in Colab.
import glob, os
folder = "BBBC005_v1_images"          # adjust to your unzip location
paths = sorted(glob.glob(os.path.join(folder, "*w1*")))   # w1 = cell-body stain
if paths:
    errors = []
    for p in paths[:50]:              # score the first 50 in-focus images
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        gt = count_from_name(os.path.basename(p))
        pred, _, _ = count_cells(img, min_area=30)
        if gt: errors.append(abs(pred - gt))
    if errors:
        print(f"scored {len(errors)} images | mean abs count error = {np.mean(errors):.1f}")
else:
    print("No BBBC005 images found locally — download from the BBBC page or Kaggle (see above).")
    print("The synthetic capstone in section 5 has already validated the pipeline end-to-end.")

---
### Module 4 complete — and the refresher series with it
Thresholding and Otsu, morphology, contours and moments, and a validated cell-counting pipeline on real
microscopy images. Convolution (Notebook 1) is the core of a CNN; segmentation/counting is the classical
counterpart of the learned segmentation in **Géron Ch. 12**, and the patch view of images appears in the
vision transformers of **Ch. 16**. Across the four modules you now have the Python, linear algebra,
calculus and image processing that Géron's book — and your Control and Robotics units — run on.